# Behavioural Planning for Autonomous Highway Driving

We plan a trajectory using the _Optimistic Planning for Deterministic systems_ ([OPD](https://hal.inria.fr/hal-00830182)) algorithm.


In [5]:
import sys
import os
import site
from importlib import reload

# Environment
!pip install highway-env
import gymnasium as gym
import highway_env

# Agent
# Install rl-agents
!pip install git+https://github.com/eleurent/rl-agents#egg=rl-agents

# Force path refresh to find the newly installed package
# This ensures /usr/local/lib/python3.12/dist-packages (or equivalent) is fully scanned
reload(site)
for path in site.getsitepackages():
    if path not in sys.path:
        sys.path.append(path)

import rl_agents
from rl_agents.agents.common.factory import agent_factory

# Visualisation
from tqdm.notebook import trange
!pip install moviepy -U
!pip install imageio_ffmpeg
!pip install pyvirtualdisplay
!apt-get install -y xvfb ffmpeg

!git clone https://github.com/Farama-Foundation/HighwayEnv.git 2>/dev/null || true
sys.path.insert(0, os.path.abspath('./HighwayEnv/scripts/'))

import utils
from utils import record_videos, show_videos

  Cloning https://github.com/eleurent/rl-agents to /tmp/pip-install-k55ft6fd/rl-agents_475ffdaf059e4a3c98ad744b9f9c89f6
  Running command git clone --filter=blob:none --quiet https://github.com/eleurent/rl-agents /tmp/pip-install-k55ft6fd/rl-agents_475ffdaf059e4a3c98ad744b9f9c89f6
  Resolved https://github.com/eleurent/rl-agents to commit 84df15ea977271e6a4d015f10f9f355f7e866890
  Preparing metadata (setup.py) ... done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


### Custom Tiny Recursive Agent
Define your recursive model and agent logic below. You can replace the `recursive_decision` logic with your specific implementation.

In [6]:
#@title Run an episode

# Make environment
env = gym.make("highway-fast-v0", render_mode="rgb_array")
env = record_videos(env)
(obs, info), done = env.reset(), False

# Make agent
agent_config = {
    "__class__": "<class 'rl_agents.agents.tree_search.deterministic.DeterministicPlannerAgent'>",
    "env_preprocessors": [{"method":"simplify"}],
    "budget": 150,
    "gamma": 0.7,
}
agent = agent_factory(env, agent_config)

# Run episode
for step in trange(env.unwrapped.config["duration"], desc="Running..."):
    action = agent.act(obs)
    obs, reward, done, truncated, info = env.step(action)

env.close()
show_videos()

Running...:   0%|          | 0/30 [00:00<?, ?it/s]

### Run Episode with Custom Recursive Agent
This cell initializes the environment and uses the `TinyRecursiveAgent` for the decision-making phase.

In [7]:
class TinyRecursiveAgent:
    def __init__(self, env):
        self.env = env
        # Initialize your recursive model parameters here

    def recursive_decision(self, obs, depth=0):
        # Base case: limit recursion depth
        if depth > 3:
            return 1 # Default action (e.g., maintain speed/lane)

        # Placeholder for recursive logic:
        # In a real scenario, you might evaluate potential future states here.
        # For now, we sample a random action as a placeholder.
        return self.env.action_space.sample()

    def act(self, obs):
        return self.recursive_decision(obs)

In [8]:
env = gym.make("highway-fast-v0", render_mode="rgb_array")
env = record_videos(env)
(obs, info), done = env.reset(), False

# Instantiate the custom recursive agent
custom_agent = TinyRecursiveAgent(env)

for step in trange(env.unwrapped.config["duration"], desc="Running Custom Agent..."):
    action = custom_agent.act(obs)
    obs, reward, done, truncated, info = env.step(action)
    if done or truncated:
        break

env.close()
show_videos()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Running Custom Agent...:   0%|          | 0/30 [00:00<?, ?it/s]

In [33]:
import os
import random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gymnasium as gym
import highway_env


def flatten_obs(obs):
    """
    highway-env observations are usually NumPy arrays.
    This function converts any observation into a flat vector.
    """
    if isinstance(obs, dict):
        parts = []
        for value in obs.values():
            parts.append(np.asarray(value, dtype=np.float32).ravel())
        return np.concatenate(parts)

    return np.asarray(obs, dtype=np.float32).ravel()


class HRMNetwork(nn.Module):
    """
    Hierarchical Reasoning Model for Q-learning.

    zL = low-level reasoning state
    zH = high-level reasoning state

    The final action_head gives Q-values:
        Q(s, action)
    """

    def __init__(self, obs_dim, action_dim, hidden_dim=128, reasoning_steps=4, T=2):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.reasoning_steps = reasoning_steps
        self.T = T

        self.obs_encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        self.L_net = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.H_net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.l_norm = nn.LayerNorm(hidden_dim)
        self.h_norm = nn.LayerNorm(hidden_dim)

        self.action_head = nn.Linear(hidden_dim, action_dim)

        # Optional ACT-style head: [halt_score, continue_score]
        self.compute_head = nn.Linear(hidden_dim, 2)

    def initial_state(self, batch_size, device):
        zH = torch.zeros(batch_size, self.hidden_dim, device=device)
        zL = torch.zeros(batch_size, self.hidden_dim, device=device)
        return zH, zL

    def forward(self, obs, z=None, adaptive=False):
        """
        obs shape: [batch, obs_dim]
        returns:
            new_z, action_q_values, compute_q_values
        """

        if obs.dim() > 2:
            obs = obs.view(obs.size(0), -1)

        batch_size = obs.size(0)
        device = obs.device

        x = self.obs_encoder(obs)

        if z is None:
            zH, zL = self.initial_state(batch_size, device)
        else:
            zH, zL = z

        for i in range(self.reasoning_steps):
            # Low-level update
            l_input = torch.cat([zL, zH, x], dim=-1)
            zL = self.l_norm(zL + torch.tanh(self.L_net(l_input)))

            # High-level update every T steps
            if ((i + 1) % self.T == 0) or (i == self.reasoning_steps - 1):
                h_input = torch.cat([zH, zL], dim=-1)
                zH = self.h_norm(zH + torch.tanh(self.H_net(h_input)))

            # Optional adaptive stopping during inference
            if adaptive and batch_size == 1 and i >= 1:
                compute_q = self.compute_head(zH)
                halt_score = compute_q[0, 0].item()
                continue_score = compute_q[0, 1].item()

                if halt_score > continue_score:
                    break

        action_q = self.action_head(zH)
        compute_q = self.compute_head(zH)

        return (zH, zL), action_q, compute_q


Transition = namedtuple(
    "Transition",
    ["obs", "action", "reward", "next_obs", "done"]
)


class ReplayBuffer:
    def __init__(self, capacity=50_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, obs, action, reward, next_obs, done):
        self.buffer.append(
            Transition(
                flatten_obs(obs),
                int(action),
                float(reward),
                flatten_obs(next_obs),
                float(done)
            )
        )

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


class HRMDQNAgent:
    def __init__(
        self,
        obs_dim,
        action_dim,
        hidden_dim=128,
        lr=1e-4,
        gamma=0.99,
        device=None
    ):
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.gamma = gamma

        self.device = device or torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.policy_net = HRMNetwork(
            obs_dim=obs_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim,
            reasoning_steps=4,
            T=2
        ).to(self.device)

        self.target_net = HRMNetwork(
            obs_dim=obs_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim,
            reasoning_steps=4,
            T=2
        ).to(self.device)

        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.AdamW(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer()

        self.train_steps = 0

    def act(self, obs, epsilon=0.05):
        """
        Epsilon-greedy action selection.
        """
        if random.random() < epsilon:
            return random.randrange(self.action_dim)

        obs_vec = flatten_obs(obs)
        obs_tensor = torch.tensor(
            obs_vec,
            dtype=torch.float32,
            device=self.device
        ).unsqueeze(0)

        with torch.no_grad():
            _, q_values, _ = self.policy_net(obs_tensor, adaptive=True)

        action = q_values.argmax(dim=1).item()
        return int(action)

    def remember(self, obs, action, reward, next_obs, done):
        self.memory.push(obs, action, reward, next_obs, done)

    def train_step(self, batch_size=64, target_update_interval=500):
        if len(self.memory) < batch_size:
            return None

        batch = self.memory.sample(batch_size)

        obs_batch = torch.tensor(
            np.stack([t.obs for t in batch]),
            dtype=torch.float32,
            device=self.device
        )

        action_batch = torch.tensor(
            [t.action for t in batch],
            dtype=torch.long,
            device=self.device
        ).unsqueeze(1)

        reward_batch = torch.tensor(
            [t.reward for t in batch],
            dtype=torch.float32,
            device=self.device
        )

        next_obs_batch = torch.tensor(
            np.stack([t.next_obs for t in batch]),
            dtype=torch.float32,
            device=self.device
        )

        done_batch = torch.tensor(
            [t.done for t in batch],
            dtype=torch.float32,
            device=self.device
        )

        _, q_values, compute_q = self.policy_net(obs_batch)

        current_q = q_values.gather(1, action_batch).squeeze(1)

        with torch.no_grad():
            _, next_q_values, _ = self.target_net(next_obs_batch)
            max_next_q = next_q_values.max(dim=1).values

            target_q = reward_batch + self.gamma * (1.0 - done_batch) * max_next_q

        dqn_loss = F.smooth_l1_loss(current_q, target_q)

        # Small optional ACT-style auxiliary loss.
        # Since highway-env has no y_true action label, we use confidence as a proxy.
        action_confidence = F.softmax(q_values.detach(), dim=1).max(dim=1).values
        halt_target = (action_confidence > 0.65).float()

        halt_logit = compute_q[:, 0] - compute_q[:, 1]
        act_loss = F.binary_cross_entropy_with_logits(halt_logit, halt_target)

        loss = dqn_loss + 0.01 * act_loss

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        self.train_steps += 1

        if self.train_steps % target_update_interval == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return loss.item()

In [ ]:
def make_highway_env(render_mode=None):
    env = gym.make("highway-fast-v0", render_mode=render_mode)

    env.unwrapped.configure({
        "observation": {
            "type": "Kinematics",
            "vehicles_count": 15,
            "features": ["presence", "x", "y", "vx", "vy"],
            "absolute": False,
            "normalize": True,
        },
        "action": {
            "type": "DiscreteMetaAction"
        },

        "duration": 40,
        "vehicles_count": 25,
        "controlled_vehicles": 1,

        "simulation_frequency": 15,
        "policy_frequency": 5,

        # safer reward settings
        "collision_reward": -20,
        "right_lane_reward": 0.05,
        "high_speed_reward": 0.05,
        "lane_change_reward": -1.0,
        "reward_speed_range": [20, 27],
    })
    return env


env = make_highway_env(render_mode=None)

obs, info = env.reset()

obs_dim = flatten_obs(obs).shape[0]
action_dim = env.action_space.n

print("Observation dimension:", obs_dim)
print("Number of actions:", action_dim)

agent = HRMDQNAgent(
    obs_dim=obs_dim,
    action_dim=action_dim,
    hidden_dim=128,
    lr=1e-4,
    gamma=0.99
)

num_episodes = 1000
max_steps_per_episode = int(
    env.unwrapped.config["duration"] * env.unwrapped.config["policy_frequency"]
)

epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay = 0.985

for episode in range(1, num_episodes + 1):
    obs, info = env.reset()

    episode_reward = 0.0
    losses = []

    epsilon = max(
        epsilon_end,
        epsilon_start * (epsilon_decay ** episode)
    )

    for step in range(max_steps_per_episode):
        action = agent.act(obs, epsilon=epsilon)

        next_obs, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        agent.remember(obs, action, reward, next_obs, done)

        loss = agent.train_step(batch_size=64)

        if loss is not None:
            losses.append(loss)

        obs = next_obs
        episode_reward += reward

        if done:
            break

    if episode % 10 == 0:
        avg_loss = np.mean(losses) if losses else 0.0
        print(
            f"Episode {episode:4d} | "
            f"Reward: {episode_reward:8.2f} | "
            f"Epsilon: {epsilon:.3f} | "
            f"Loss: {avg_loss:.4f}"
        )

env.close()

Observation dimension: 75
Number of actions: 5
Episode   10 | Reward:    12.97 | Epsilon: 0.860 | Loss: 0.0401
Episode   20 | Reward:    14.96 | Epsilon: 0.739 | Loss: 0.0847
Episode   30 | Reward:    53.81 | Epsilon: 0.635 | Loss: 0.0675
Episode   40 | Reward:    21.94 | Epsilon: 0.546 | Loss: 0.0902
Episode   50 | Reward:    42.90 | Epsilon: 0.470 | Loss: 0.1872
Episode   60 | Reward:    17.97 | Epsilon: 0.404 | Loss: 0.1251
Episode   70 | Reward:    88.80 | Epsilon: 0.347 | Loss: 0.1211
Episode   80 | Reward:    13.99 | Epsilon: 0.298 | Loss: 0.1268
Episode   90 | Reward:    25.96 | Epsilon: 0.257 | Loss: 0.1510
Episode  100 | Reward:    59.97 | Epsilon: 0.221 | Loss: 0.1576
Episode  110 | Reward:    28.95 | Epsilon: 0.190 | Loss: 0.2485
Episode  120 | Reward:    75.80 | Epsilon: 0.163 | Loss: 0.2509
Episode  130 | Reward:    32.87 | Epsilon: 0.140 | Loss: 0.2172
Episode  140 | Reward:    72.77 | Epsilon: 0.121 | Loss: 0.2433
Episode  150 | Reward:    62.77 | Epsilon: 0.104 | Loss: 

In [ ]:
# Increase duration to make the video last longer
long_env = make_highway_env(render_mode="rgb_array")
long_env.unwrapped.configure({"duration": 100})  # Increased from default
long_env = record_videos(long_env)

(obs, info), done = long_env.reset(), False

for step in trange(long_env.unwrapped.config["duration"], desc="Running Longer Evaluation..."):
    action = agent.act(obs, epsilon=0)
    obs, reward, done, truncated, info = long_env.step(action)
    if done or truncated:
        break

long_env.close()
show_videos()

In [32]:
# Correctly set up the environment for visualization
# Note: we use 'rgb_array' here so it can be recorded

eval_env = make_highway_env(render_mode="rgb_array")
eval_env = record_videos(eval_env, video_folder="videos")

obs, info = eval_env.reset()
eval_steps = int(
    eval_env.unwrapped.config["duration"] * eval_env.unwrapped.config["policy_frequency"]
)

action_counts = {}

for step in range(eval_steps):
    action = agent.act(obs, epsilon=0.0)

    action_counts[action] = action_counts.get(action, 0) + 1

    obs, reward, terminated, truncated, info = eval_env.step(action)

    if terminated or truncated:
        break

eval_env.close()
show_videos("videos")

print("Action counts:", action_counts)

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Action counts: {3: 28, 1: 4, 4: 20, 0: 74, 2: 74}


In [27]:
test_env = gym.make("highway-v0", render_mode="human")

obs, info = test_env.reset()

for step in range(500):
    action = agent.act(obs, epsilon=0.0)

    obs, reward, terminated, truncated, info = test_env.step(action)

    if terminated or truncated:
        obs, info = test_env.reset()

test_env.close()